In [1]:
import pandas as pd
import numpy as np

doc = pd.read_csv("D:\\Shukra_sir\\MLAllTraining\\ForTraining\\TrainableWithBandRatios\\DOC_BR.csv")
df = doc
df.columns

Index(['sample_date', 'B1', 'B11', 'B12', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7',
       'B8', 'B8A', 'B9', 'datetime_utc', 'time_difference_hours', 'latitude',
       'longitude', 'sampling_year', 'sampling_month', 'sampling_day',
       'sampling_hour', 'sampling_minute', 'sampling_second', 'Temperature',
       'DewPoint', 'v10n', 'Precipitation(mm)', 'sample_date_utc',
       'latitude_with', 'longitude_with', 'result_DOC',
       'DOC_Merged_How_many_n', 'county_name', 'sample_code', 'parameter',
       'sample_depth', 'sample_depth_units', 'reporting_limit', 'units',
       'method_name', 'B4_minus_B3', 'B2_minus_B4', 'B5_minus_B2', 'B4_div_B3',
       'B6_div_B8', 'B3_div_B2', 'ND_B4_B3', 'ND_B2_B3', 'ND_B6_B8'],
      dtype='str')

In [2]:
#Features

"""
"For DOC only"
top_BR1 = [
    "B4_minus_B3",
    "B2_minus_B4",
    "B5_minus_B2"
]

top_BR2 = [
    "B4_div_B3",
    "B6_div_B8",
    "B3_div_B2"
]

top_BR3 = [
    "ND_B4_B3",
    "ND_B2_B3",
    "ND_B6_B8"
]
"""

#band_features = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 
#               "B4_minus_B3", "B2_minus_B4", "B5_minus_B2",
#               "B4_div_B3", "B6_div_B8", "B3_div_B2", 
#               "ND_B4_B3", "ND_B2_B3", "ND_B6_B8"]

band_features = ['B1','B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12']


meteorological_features = ['Temperature', 'DewPoint', 'v10n', 'Precipitation(mm)']

continuous_features = meteorological_features + band_features

static_features = ["latitude", "longitude", "doy_sin", "doy_cos", "hour_sin", "hour_cos"]

county_col = "county_name"

top_BR1 = [
    "B4_minus_B3",
    "B2_minus_B4",
    "B5_minus_B2"
]

top_BR2 = [
    "B4_div_B3",
    "B6_div_B8",
    "B3_div_B2"
]

top_BR3 = [
    "ND_B4_B3",
    "ND_B2_B3",
    "ND_B6_B8"
]

ALL_RATIOS = top_BR1 + top_BR2 + top_BR3


In [3]:
random_state = 42
desired_train_size = 2500 # optional, not enforced here

NOISE_CONFIG = {
    "Temperature": ("additive", 0.3),
    "DewPoint": ("additive", 0.3),
    "v10n": ("multiplicative", 0.05),
    "Precipitation(mm)": ("multiplicative", 0.010)
}

for br in band_features:
    NOISE_CONFIG[br] = ("multiplicative", 0.02)


In [4]:
# -------------------------------
#  CYCLIC ENCODING FOR DATE/TIME
# -------------------------------
import numpy as np

df["sample_date_utc"] = pd.to_datetime(df["sample_date_utc"])
df["day_of_year"] = df["sample_date_utc"].dt.dayofyear

df["doy_sin"] = np.sin(2 * np.pi * df["day_of_year"] / 365)
df["doy_cos"] = np.cos(2 * np.pi * df["day_of_year"] / 365)

df["hour_sin"] = np.sin(2 * np.pi * df["sampling_hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["sampling_hour"] / 24)

In [34]:
#features_initial = ['sample_date_utc', 'sampling_month', 'sampling_day', 'sampling_hour',
#                    'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 
#                    "B4_minus_B3", "B2_minus_B4", "B5_minus_B2",
#                    "B4_div_B3", "B6_div_B8", "B3_div_B2", 
#                    "ND_B4_B3", "ND_B2_B3", "ND_B6_B8",
#                    'Temperature', 'DewPoint', 'v10n', "Precipitation(mm)",
#                    'doy_sin', 'doy_cos', 'hour_sin', 'hour_cos', "county_name", 'latitude', 'longitude',
#                    "result_DOC"]

In [5]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=random_state,
    shuffle=True
)

train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

In [6]:
train_df.shape

(683, 54)

In [7]:
#NOISE INJECTION

def apply_noise(df, noise_config, random_state=42):
    rng = np.random.default_rng(random_state)
    noisy_df = df.copy()

    for col, (mode, strength) in noise_config.items():
        if col not in df.columns:
            continue

        if mode == "additive":
            noise = rng.normal(0, strength, size=len(df))
            noisy_df[col] = df[col] + noise

        elif mode == "multiplicative":
            noise = rng.normal(0, strength, size=len(df))
            noisy_df[col] = df[col] * (1 + noise)

    return noisy_df

In [8]:
def recompute_band_ratios(df):
    df = df.copy()
    eps = 1e-6

    # Differences
    df["B4_minus_B3"] = df["B4"] - df["B3"]
    df["B2_minus_B4"] = df["B2"] - df["B4"]
    df["B5_minus_B2"] = df["B5"] - df["B2"]

    # Ratios
    df["B4_div_B3"] = df["B4"] / (df["B3"] + eps)
    df["B6_div_B8"] = df["B6"] / (df["B8"] + eps)
    df["B3_div_B2"] = df["B3"] / (df["B2"] + eps)

    # Normalized differences
    df["ND_B4_B3"] = (df["B4"] - df["B3"]) / (df["B4"] + df["B3"] + eps)
    df["ND_B2_B3"] = (df["B2"] - df["B3"]) / (df["B2"] + df["B3"] + eps)
    df["ND_B6_B8"] = (df["B6"] - df["B8"]) / (df["B6"] + df["B8"] + eps)

    return df


In [9]:
AUG_FACTOR = 3   # increase carefully

augmented_dfs = [train_df]

for i in range(AUG_FACTOR):
    noisy = apply_noise(
        train_df,
        noise_config=NOISE_CONFIG,
        random_state=random_state + i
    )

    noisy = recompute_band_ratios(noisy)

    augmented_dfs.append(noisy)

train_augmented = pd.concat(augmented_dfs, ignore_index=True)


In [64]:
FilePath_train = f"D:\Shukra_sir\MLAllTraining\ForTraining\TrainableWithBandRatios\AugmentedWithBR\DOC_AUG_WITH_BR\DOCWithBR_TRAIN_AUG{AUG_FACTOR}.csv"

train_augmented.to_csv(FilePath_train)
print(f"Saved Training data to folder {FilePath_train}")



Saved Training data to folder D:\Shukra_sir\MLAllTraining\ForTraining\TrainableWithBandRatios\AugmentedWithBR\DOC_AUG_WITH_BR\DOCWithBR_TRAIN_AUG3.csv


In [65]:
FilePath_test = f"D:\Shukra_sir\MLAllTraining\ForTraining\TrainableWithBandRatios\AugmentedWithBR\DOC_AUG_WITH_BR\DOCWithBR_TEST_AUG{AUG_FACTOR}.csv"

test_df.to_csv(FilePath_test)
print(f"Saved Testing data to folder {FilePath_test}")

Saved Testing data to folder D:\Shukra_sir\MLAllTraining\ForTraining\TrainableWithBandRatios\AugmentedWithBR\DOC_AUG_WITH_BR\DOCWithBR_TEST_AUG3.csv


In [10]:
FEATURES = (
    continuous_features +
    static_features +
    ALL_RATIOS
)
FEATURES

['Temperature',
 'DewPoint',
 'v10n',
 'Precipitation(mm)',
 'B1',
 'B2',
 'B3',
 'B4',
 'B5',
 'B6',
 'B7',
 'B8',
 'B8A',
 'B9',
 'B11',
 'B12',
 'latitude',
 'longitude',
 'doy_sin',
 'doy_cos',
 'hour_sin',
 'hour_cos',
 'B4_minus_B3',
 'B2_minus_B4',
 'B5_minus_B2',
 'B4_div_B3',
 'B6_div_B8',
 'B3_div_B2',
 'ND_B4_B3',
 'ND_B2_B3',
 'ND_B6_B8']

In [11]:
FEATURES = (
    continuous_features +
    static_features +
    ALL_RATIOS
)

TARGET = "result_DOC"

X_train = train_augmented[FEATURES]
y_train = train_augmented[TARGET]

X_test = test_df[FEATURES]
y_test = test_df[TARGET]


In [ ]:
#EVALUATION 

In [ ]:
stats_orig = train_df[continuous_features].describe().T
stats_aug  = train_augmented[continuous_features].describe().T

print(pd.concat([stats_orig[['mean','std', '25%', '50%', '75%']], stats_aug[['mean','std','25%', '50%', '75%' ]]], axis=1, keys=['Original','Augmented']))


#Expect mean close to original, std slightly higher due to added noise


                     Original             Augmented          
                         mean       std        mean       std
Temperature        292.681184  6.976439  292.683201  6.973901
DewPoint           279.040670  6.615643  279.038320  6.616330
v10n                -0.607352  1.874802   -0.606516  1.871616
Precipitation(mm)    0.000219  0.002255    0.000219  0.002256
B1                   0.049983  0.033884    0.050023  0.033938
B2                   0.061681  0.043733    0.061687  0.043729
B3                   0.081682  0.049723    0.081685  0.049694
B4                   0.084500  0.062594    0.084467  0.062576
B5                   0.102520  0.068530    0.102566  0.068546
B6                   0.125187  0.084801    0.125242  0.084814
B7                   0.136396  0.092998    0.136305  0.092885
B8                   0.140919  0.100035    0.140931  0.100053
B8A                  0.143622  0.100863    0.143580  0.100776
B9                   0.166818  0.091760    0.166686  0.091714
B11     

In [12]:
stats_orig = train_df[continuous_features].describe().T
stats_aug  = train_augmented[continuous_features].describe().T

print(pd.concat([stats_orig[['mean','std', '25%', '50%', '75%']], stats_aug[['mean','std','25%', '50%', '75%' ]]], axis=1, keys=['Original','Augmented']))


#Expect mean close to original, std slightly higher due to added noise

                     Original                                                \
                         mean       std         25%         50%         75%   
Temperature        292.681184  6.976439  287.476691  292.927241  297.581426   
DewPoint           279.040670  6.615643  274.159255  279.751447  284.696858   
v10n                -0.607352  1.874802   -1.711883   -0.484027    0.791018   
Precipitation(mm)    0.000219  0.002255    0.000000    0.000000    0.000000   
B1                   0.049983  0.033884    0.027600    0.043200    0.067000   
B2                   0.061681  0.043733    0.030500    0.050300    0.082350   
B3                   0.081682  0.049723    0.045700    0.069300    0.104000   
B4                   0.084500  0.062594    0.039050    0.065200    0.110500   
B5                   0.102520  0.068530    0.051250    0.090300    0.144000   
B6                   0.125187  0.084801    0.049550    0.124200    0.188600   
B7                   0.136396  0.092998    0.051900 

In [13]:
mad = (train_augmented[continuous_features].mean() - train_df[continuous_features].mean()).abs()
print("Mean deviation per feature:\n", mad)

#Small MAD → augmented data preserves original distribution

Mean deviation per feature:
 Temperature          2.016693e-03
DewPoint             2.350008e-03
v10n                 8.357559e-04
Precipitation(mm)    3.587080e-08
B1                   4.070135e-05
B2                   5.252158e-06
B3                   3.599431e-06
B4                   3.328319e-05
B5                   4.637751e-05
B6                   5.485669e-05
B7                   9.028197e-05
B8                   1.234231e-05
B8A                  4.155644e-05
B9                   1.315533e-04
B11                  3.033951e-05
B12                  6.943182e-06
dtype: float64


In [14]:
from scipy.stats import wasserstein_distance

for col in continuous_features:
    wd = wasserstein_distance(train_df[col], train_augmented[col])
    print(f"{col}: Wasserstein distance = {wd:.4f}")

#Smaller values → augmented data closer to original

Temperature: Wasserstein distance = 0.0452
DewPoint: Wasserstein distance = 0.0434
v10n: Wasserstein distance = 0.0125
Precipitation(mm): Wasserstein distance = nan
B1: Wasserstein distance = 0.0002
B2: Wasserstein distance = 0.0003
B3: Wasserstein distance = 0.0003
B4: Wasserstein distance = 0.0003
B5: Wasserstein distance = 0.0004
B6: Wasserstein distance = 0.0004
B7: Wasserstein distance = 0.0004
B8: Wasserstein distance = 0.0005
B8A: Wasserstein distance = 0.0005
B9: Wasserstein distance = 0.0006
B11: Wasserstein distance = 0.0005
B12: Wasserstein distance = 0.0004


In [15]:
corr_orig = train_df[continuous_features].corr()
corr_aug  = train_augmented[continuous_features].corr()

# Compare
diff_corr = (corr_orig - corr_aug).abs()
print(diff_corr)

#Expect small differences, major patterns preserved

                   Temperature  DewPoint      v10n  Precipitation(mm)  \
Temperature           0.000000  0.000674  0.000930           0.000383   
DewPoint              0.000674  0.000000  0.002188           0.000984   
v10n                  0.000930  0.002188  0.000000           0.001271   
Precipitation(mm)     0.000383  0.000984  0.001271           0.000000   
B1                    0.000274  0.002077  0.000894           0.001844   
B2                    0.002041  0.000321  0.000242           0.000386   
B3                    0.000570  0.000412  0.001284           0.000206   
B4                    0.001140  0.000141  0.000061           0.000637   
B5                    0.001591  0.000014  0.000710           0.000720   
B6                    0.000880  0.001248  0.000023           0.001626   
B7                    0.000081  0.000570  0.000478           0.000389   
B8                    0.000618  0.001525  0.000499           0.001319   
B8A                   0.000225  0.000810  0.000841 

In [ ]:
#5️⃣ Visual check by county

#Since you do county-aware augmentation, plot feature distributions per county:

import seaborn as sns
import matplotlib.pyplot as plt

for county in train_df[county_col].unique():
    sns.kdeplot(train_df[train_df[county_col]==county]['Temperature'], label='Original')
    sns.kdeplot(train_aug[train_aug[county_col]==county]['Temperature'], label='Augmented')
    plt.title(f'Temperature distribution - {county}')
    plt.legend()
    plt.show()

#Ensures small counties are not distorted by oversampling
